# Qwen3.5-9B 640/768 Multi-scale + Flip TTA

학습은 하지 않습니다. Drive에 저장된 640 best adapter를 불러와 다음 validation/test
확률을 생성합니다.

1. 기존 640 원본 확률
2. 640 horizontal flip
3. 768 원본
4. 768 horizontal flip
5. 640 flip TTA, 640/768 multi-scale, 768 flip TTA, 전체 TTA

`왼쪽/오른쪽/좌측/우측` 질문은 flip하지 않습니다. A100 80GB 런타임에서 **모두 실행**하세요.
예상 시간은 test 세 번 추론을 포함해 약 35~55분입니다.


## 1. 환경 설치


In [ ]:
import sys, subprocess, importlib
import importlib.metadata as metadata

TARGET_VERSIONS = {"transformers": "5.15.1", "peft": "0.20.0", "Pillow": "11.3.0"}

def installed_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None

torchao_version = installed_version("torchao")
if torchao_version is not None:
    assert "torchao" not in sys.modules, "런타임을 삭제하고 다시 모두 실행하세요."
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
    importlib.invalidate_caches()

before = {name: installed_version(name) for name in TARGET_VERSIONS}
if not all(before[name] == version for name, version in TARGET_VERSIONS.items()):
    preloaded = [name for name in ("transformers", "peft", "PIL") if name in sys.modules]
    if preloaded:
        raise RuntimeError(f"패키지가 이미 import되었습니다: {preloaded}. 런타임을 삭제하세요.")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--upgrade",
        "transformers==5.15.1", "peft==0.20.0", "accelerate>=1.14.0",
        "safetensors>=0.6.0", "Pillow==11.3.0", "pandas>=2.2", "tqdm>=4.66",
    ], check=True)

after = {name: installed_version(name) for name in TARGET_VERSIONS}
assert after == TARGET_VERSIONS, after
print("환경 준비 완료:", after)


## 2. 설정, 데이터, 640 source run 자동 탐색


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc, json, shutil, zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from tqdm.auto import tqdm

assert torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert GPU_VRAM_GIB >= 70, f"A100 80GB 필요: {GPU_NAME}, {GPU_VRAM_GIB:.1f} GiB"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")
Image.MAX_IMAGE_PIXELS = None

MODEL_ID = "Qwen/Qwen3.5-9B"
DATA_ROOT = Path("/content")
DATA_ARCHIVE = Path("/content/drive/MyDrive/2026-ssafy-15-2-ai.zip")
DRIVE_SOURCE_ROOT = Path("/content/drive/MyDrive/qwen35_9b_640")
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/qwen35_9b_tta")

# 특정 실행 폴더를 쓰고 싶다면 None 대신 timestamp 폴더 경로를 넣으세요.
SOURCE_RUN_DIR = None
INFER_BATCH_SIZE = 8
RUN_TEST = True
EXPORT_TO_DRIVE = True

LETTERS = ["a", "b", "c", "d"]
LETTER_TO_INDEX = {letter: index for index, letter in enumerate(LETTERS)}
PROB_COLUMNS = [f"prob_{letter}" for letter in LETTERS]
RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_ROOT = Path("/content/qwen35_tta") / RUN_STAMP
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def clear_cuda():
    gc.collect()
    torch.cuda.empty_cache()

from google.colab import drive
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

required = [
    DATA_ROOT / "train.csv", DATA_ROOT / "test.csv",
    DATA_ROOT / "train", DATA_ROOT / "test",
]
if not all(path.exists() for path in required):
    assert DATA_ARCHIVE.exists(), DATA_ARCHIVE
    with zipfile.ZipFile(DATA_ARCHIVE) as archive:
        archive.extractall(DATA_ROOT)
assert all(path.exists() for path in required)

train_df = pd.read_csv(DATA_ROOT / "train.csv")
test_df = pd.read_csv(DATA_ROOT / "test.csv")

def categorize(question):
    question = str(question)
    if "몇 개" in question or "개수" in question:
        return "counting"
    if "재질" in question or "소재" in question:
        return "material"
    if "색" in question:
        return "color"
    if "종류" in question:
        return "type"
    return "other"

for frame in (train_df, test_df):
    frame["category"] = frame["question"].map(categorize)
valid_df = train_df.iloc[int(len(train_df) * 0.9):].copy().reset_index(drop=True)
assert len(valid_df) == 508 and len(test_df) == 5074

def image_path(relative_path):
    path = Path(str(relative_path))
    return path if path.is_absolute() else DATA_ROOT / path

if SOURCE_RUN_DIR is None:
    candidates = []
    if DRIVE_SOURCE_ROOT.exists():
        for adapter_config in DRIVE_SOURCE_ROOT.glob("*/best_adapter/adapter_config.json"):
            run_dir = adapter_config.parent.parent
            valid_file = run_dir / "qwen35_640_best_valid.csv"
            test_file = run_dir / "qwen35_640_best_test.csv"
            if valid_file.exists() and test_file.exists():
                candidates.append(run_dir)
    assert candidates, f"640 source run을 찾지 못했습니다: {DRIVE_SOURCE_ROOT}"
    SOURCE_RUN_DIR = sorted(candidates, key=lambda path: path.name)[-1]
else:
    SOURCE_RUN_DIR = Path(SOURCE_RUN_DIR)

SOURCE_ADAPTER_DIR = SOURCE_RUN_DIR / "best_adapter"
SOURCE_VALID_PATH = SOURCE_RUN_DIR / "qwen35_640_best_valid.csv"
SOURCE_TEST_PATH = SOURCE_RUN_DIR / "qwen35_640_best_test.csv"
for path in (SOURCE_ADAPTER_DIR, SOURCE_VALID_PATH, SOURCE_TEST_PATH):
    assert path.exists(), path

original640_valid = pd.read_csv(SOURCE_VALID_PATH)
original640_test = pd.read_csv(SOURCE_TEST_PATH)
assert original640_valid["id"].tolist() == valid_df["id"].tolist()
assert original640_test["id"].tolist() == test_df["id"].tolist()
print("GPU:", GPU_NAME, f"{GPU_VRAM_GIB:.1f} GiB")
print("source run:", SOURCE_RUN_DIR)
print("output:", OUTPUT_ROOT)


## 3. 모델, adapter, 640/768 processor 로드


In [ ]:
from transformers import AutoProcessor
try:
    from transformers import Qwen3_5ForConditionalGeneration as Qwen35Model
except ImportError:
    from transformers import AutoModelForMultimodalLM as Qwen35Model
from peft import PeftModel

def make_processor(size):
    processor = AutoProcessor.from_pretrained(
        MODEL_ID, min_pixels=size * size, max_pixels=size * size, trust_remote_code=True,
    )
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.padding_side = "left"
    return processor

processor640 = make_processor(640)
processor768 = make_processor(768)

base_model = Qwen35Model.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0}, low_cpu_mem_usage=True,
    trust_remote_code=True, attn_implementation="sdpa",
)
model = PeftModel.from_pretrained(base_model, str(SOURCE_ADAPTER_DIR), is_trainable=False)
model.eval()
MODEL_DEVICE = next(model.parameters()).device
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

SYSTEM_INSTRUCTION = (
    "You are an expert visual multiple-choice question answering system. "
    "Inspect the entire image carefully. For quantity questions, count every relevant visible object exactly once. "
    "Answer with exactly one lowercase letter: a, b, c, or d. Do not explain."
)

def build_prompt(row):
    return (
        f"{row['question']}\n(a) {row['a']}\n(b) {row['b']}\n"
        f"(c) {row['c']}\n(d) {row['d']}\n\n"
        "정답을 a, b, c, d 중 한 글자로만 출력하세요."
    )

def build_messages(row, image):
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCTION}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": build_prompt(row)},
        ]},
    ]

def apply_template(processor, messages):
    kwargs = dict(tokenize=False, add_generation_prompt=True)
    try:
        return processor.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:
        return processor.apply_chat_template(messages, **kwargs)

dummy = [
    {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCTION}]},
    {"role": "user", "content": [{"type": "text", "text": "Choose: (a) A (b) B (c) C (d) D"}]},
]
prefix = apply_template(processor640, dummy)
prefix_ids = processor640.tokenizer(prefix, add_special_tokens=False)["input_ids"]
LETTER_TOKEN_IDS = {}
for letter in LETTERS:
    extended = processor640.tokenizer(prefix + letter, add_special_tokens=False)["input_ids"]
    assert extended[:len(prefix_ids)] == prefix_ids
    LETTER_TOKEN_IDS[letter] = extended[len(prefix_ids)]
assert len(set(LETTER_TOKEN_IDS.values())) == 4
print("model loaded / letter tokens:", LETTER_TOKEN_IDS)


## 4. TTA 추론 및 확률 결합 함수


In [ ]:
DIRECTION_TERMS = ("왼쪽", "오른쪽", "좌측", "우측")

def has_direction_question(row):
    return any(term in str(row["question"]) for term in DIRECTION_TERMS)

def score_view(dataframe, processor, flip, desc):
    probabilities = []
    letter_tensor = torch.tensor([LETTER_TOKEN_IDS[x] for x in LETTERS], device=MODEL_DEVICE)
    model.eval()
    with torch.inference_mode():
        for start in tqdm(range(0, len(dataframe), INFER_BATCH_SIZE), desc=desc, unit="batch"):
            chunk = dataframe.iloc[start:start + INFER_BATCH_SIZE]
            images, texts = [], []
            for _, row in chunk.iterrows():
                with Image.open(image_path(row["path"])) as opened:
                    image = ImageOps.exif_transpose(opened).convert("RGB")
                if flip and not has_direction_question(row):
                    image = ImageOps.mirror(image)
                images.append(image)
                texts.append(apply_template(processor, build_messages(row, image)))
            inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(MODEL_DEVICE)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                try:
                    outputs = model(**inputs, use_cache=False, logits_to_keep=1)
                except TypeError:
                    outputs = model(**inputs, use_cache=False)
            logits = outputs.logits[:, -1, :].index_select(-1, letter_tensor)
            probs = torch.softmax(logits.float(), dim=-1)
            probabilities.extend(probs.cpu().tolist())
            del inputs, outputs, logits, probs, images, texts
    result = dataframe.reset_index(drop=True).copy()
    matrix = np.asarray(probabilities, dtype=np.float64)
    for index, letter in enumerate(LETTERS):
        result[f"prob_{letter}"] = matrix[:, index]
    result["pred"] = [LETTERS[index] for index in matrix.argmax(axis=1)]
    result["pred_conf"] = matrix.max(axis=1)
    if "answer" in result:
        result["correct"] = result["pred"] == result["answer"]
    return result

def probs(frame):
    values = frame[PROB_COLUMNS].to_numpy(dtype=np.float64)
    return values / values.sum(axis=1, keepdims=True)

def log_average(frames):
    matrices = [probs(frame) for frame in frames]
    score = np.mean([np.log(np.clip(matrix, 1e-12, 1.0)) for matrix in matrices], axis=0)
    score -= score.max(axis=1, keepdims=True)
    values = np.exp(score)
    return values / values.sum(axis=1, keepdims=True)

def scored_from_probability(reference, probability):
    result = reference.copy()
    for index, letter in enumerate(LETTERS):
        result[f"prob_{letter}"] = probability[:, index]
    result["pred"] = [LETTERS[index] for index in probability.argmax(axis=1)]
    result["pred_conf"] = probability.max(axis=1)
    if "answer" in result:
        result["correct"] = result["pred"] == result["answer"]
    return result

def metrics(frame):
    correct = int(frame["correct"].sum())
    matrix = probs(frame)
    gold = frame["answer"].map(LETTER_TO_INDEX).to_numpy()
    nll = float(-np.log(np.clip(matrix[np.arange(len(frame)), gold], 1e-12, 1.0)).mean())
    return correct, nll


## 5. Validation TTA 비교


In [ ]:
flip640_valid = score_view(valid_df, processor640, True, "valid 640 flip")
original768_valid = score_view(valid_df, processor768, False, "valid 768 original")
flip768_valid = score_view(valid_df, processor768, True, "valid 768 flip")

valid_candidates = {
    "original640": original640_valid,
    "flip640": flip640_valid,
    "original768": original768_valid,
    "flip768": flip768_valid,
    "tta640": scored_from_probability(
        original640_valid, log_average([original640_valid, flip640_valid])
    ),
    "multiscale_original": scored_from_probability(
        original640_valid, log_average([original640_valid, original768_valid])
    ),
    "tta768": scored_from_probability(
        original640_valid, log_average([original768_valid, flip768_valid])
    ),
    "full_tta": scored_from_probability(
        original640_valid,
        log_average([original640_valid, flip640_valid, original768_valid, flip768_valid]),
    ),
}

rows = []
for name, frame in valid_candidates.items():
    correct, nll = metrics(frame)
    row = {"candidate": name, "correct": correct, "nll": nll}
    for category, group in frame.groupby("category"):
        row[category] = int(group["correct"].sum())
    rows.append(row)
validation_table = pd.DataFrame(rows).sort_values(
    ["correct", "nll"], ascending=[False, True]
).reset_index(drop=True)
validation_table.to_csv(OUTPUT_ROOT / "tta_validation_summary.csv", index=False)
print(validation_table.to_string(index=False))

best_candidate = str(validation_table.iloc[0]["candidate"])
best_valid = valid_candidates[best_candidate]
best_valid.to_csv(OUTPUT_ROOT / "qwen35_tta_best_valid.csv", index=False)

gold = valid_df["answer"].to_numpy()
oracle = np.zeros(len(valid_df), dtype=bool)
for frame in valid_candidates.values():
    oracle |= frame["pred"].to_numpy() == gold
print("selected:", best_candidate, "/ oracle:", int(oracle.sum()), "/508")


## 6. Test TTA와 모든 확률 CSV 저장


In [ ]:
if RUN_TEST:
    flip640_test = score_view(test_df, processor640, True, "test 640 flip")
    original768_test = score_view(test_df, processor768, False, "test 768 original")
    flip768_test = score_view(test_df, processor768, True, "test 768 flip")

    test_candidates = {
        "original640": original640_test,
        "flip640": flip640_test,
        "original768": original768_test,
        "flip768": flip768_test,
        "tta640": scored_from_probability(
            original640_test, log_average([original640_test, flip640_test])
        ),
        "multiscale_original": scored_from_probability(
            original640_test, log_average([original640_test, original768_test])
        ),
        "tta768": scored_from_probability(
            original640_test, log_average([original768_test, flip768_test])
        ),
        "full_tta": scored_from_probability(
            original640_test,
            log_average([original640_test, flip640_test, original768_test, flip768_test]),
        ),
    }

    for name, frame in valid_candidates.items():
        frame.to_csv(OUTPUT_ROOT / f"qwen35_tta_{name}_valid.csv", index=False)
    for name, frame in test_candidates.items():
        frame.to_csv(OUTPUT_ROOT / f"qwen35_tta_{name}_test.csv", index=False)
        submission = frame[["id", "pred"]].rename(columns={"pred": "answer"})
        assert len(submission) == 5074 and submission["answer"].isin(LETTERS).all()
        submission.to_csv(OUTPUT_ROOT / f"submission_qwen35_tta_{name}.csv", index=False)

    best_test = test_candidates[best_candidate]
    print("best validation candidate test saved:", best_candidate)
else:
    print("RUN_TEST=False")


## 7. Drive 백업


In [ ]:
metadata_payload = {
    "source_run": str(SOURCE_RUN_DIR),
    "adapter": str(SOURCE_ADAPTER_DIR),
    "best_candidate": best_candidate,
    "gpu": GPU_NAME,
    "validation": validation_table.to_dict(orient="records"),
}
with open(OUTPUT_ROOT / "tta_metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata_payload, file, ensure_ascii=False, indent=2)

if EXPORT_TO_DRIVE:
    drive_run = DRIVE_OUTPUT_ROOT / RUN_STAMP
    drive_run.mkdir(parents=True, exist_ok=True)
    for artifact in OUTPUT_ROOT.iterdir():
        if artifact.is_file():
            shutil.copy2(artifact, drive_run / artifact.name)
    print("Drive backup:", drive_run)

print("완료 / selected:", best_candidate)
print("local:", OUTPUT_ROOT)


## 완료 후

Drive의 `/MyDrive/qwen35_9b_tta/<실행시각>/` 폴더를 통째로 내려받아 프로젝트의
`TTA_file` 폴더에 넣으세요. 특히 `tta_validation_summary.csv`와 모든
`qwen35_tta_*_valid.csv`, `qwen35_tta_*_test.csv`가 필요합니다.
